In [0]:
spark.table(
    "workspace.transportation_analytics.gold_route_profitability"
).show(10, truncate=False)

+--------+-----------+--------------------+------------------+--------------------+-------------+---------------+-----------+------------+----------------+-----------------+----------------------+
|route_id|total_loads|total_revenue       |total_fuel_cost   |total_distance_miles|profit       |profit_per_mile|origin_city|origin_state|destination_city|destination_state|typical_distance_miles|
+--------+-----------+--------------------+------------------+--------------------+-------------+---------------+-----------+------------+----------------+-----------------+----------------------+
|RTE00006|1460       |5053135.729999995   |1639811.7400000002|2432234             |3413323.99   |1.4            |Chicago    |IL          |Charlotte       |NC               |675                   |
|RTE00014|1487       |9073774.530000022   |1654412.9300000016|4352436             |7419361.6    |1.7            |Phoenix    |AZ          |Kansas City     |MO               |1204                  |
|RTE00035|1441 

In [0]:
from pyspark.sql import functions as F

silver_routes = spark.table(
    "workspace.transportation_analytics.silver_routes"
)

silver_loads = spark.table(
    "workspace.transportation_analytics.silver_loads"
)

silver_trips = spark.table(
    "workspace.transportation_analytics.silver_trips"
)

silver_fuel_purchases = spark.table(
    "workspace.transportation_analytics.silver_fuel_purchases"
)

print("Silver route profitability tables loaded successfully")

silver_routes.printSchema()
silver_loads.printSchema()
silver_trips.printSchema()
silver_fuel_purchases.printSchema()

Silver route profitability tables loaded successfully
root
 |-- route_id: string (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- destination_state: string (nullable = true)
 |-- typical_distance_miles: integer (nullable = true)
 |-- base_rate_per_mile: double (nullable = true)
 |-- fuel_surcharge_rate: double (nullable = true)
 |-- typical_transit_days: integer (nullable = true)

root
 |-- load_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- load_type: string (nullable = true)
 |-- weight_lbs: integer (nullable = true)
 |-- pieces: integer (nullable = true)
 |-- revenue: double (nullable = true)
 |-- fuel_surcharge: double (nullable = true)
 |-- accessorial_charges: integer (nullable = true)
 |-- load_status: string (nullable = true)
 |-- booking_type: string (nullable = 

In [0]:
gold_route_profitability = (
    silver_loads
    .join(
        silver_trips.select(
            "trip_id",
            "load_id",
            "actual_distance_miles"
        ),
        on="load_id",
        how="left"
    )
    .join(
        silver_fuel_purchases.select(
            "trip_id",
            "total_cost"
        ),
        on="trip_id",
        how="left"
    )
)

print("Loads, trips, and fuel costs joined successfully")

Loads, trips, and fuel costs joined successfully


In [0]:
gold_route_profitability = (
    gold_route_profitability
    .groupBy("route_id")
    .agg(
        F.countDistinct("load_id").alias("total_loads"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("total_cost").alias("total_fuel_cost"),
        F.sum("actual_distance_miles").alias("total_distance_miles")
    )
)

print("Route-level profitability metrics calculated successfully")

Route-level profitability metrics calculated successfully


In [0]:
gold_route_profitability = (
    gold_route_profitability
    .withColumn(
        "profit",
        F.round(
            F.col("total_revenue") - F.col("total_fuel_cost"),
            2
        )
    )
    .withColumn(
        "profit_per_mile",
        F.round(
            F.col("profit") / F.col("total_distance_miles"),
            2
        )
    )
)

print("Route profitability calculated successfully")

Route profitability calculated successfully


In [0]:
gold_route_profitability = (
    gold_route_profitability
    .join(
        silver_routes.select(
            "route_id",
            "origin_city",
            "origin_state",
            "destination_city",
            "destination_state",
            "typical_distance_miles"
        ),
        on="route_id",
        how="left"
    )
)

print("Route details added to Gold profitability data")

Route details added to Gold profitability data


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_route_profitability"
)

target.alias("t").merge(
    gold_route_profitability.alias("s"),
    "t.route_id = s.route_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Route Profitability table updated using MERGE")

Gold Route Profitability table updated using MERGE


In [0]:
spark.table(
    "workspace.transportation_analytics.gold_route_profitability"
).show(10, truncate=False)

+--------+-----------+--------------------+------------------+--------------------+-------------+---------------+-----------+------------+----------------+-----------------+----------------------+
|route_id|total_loads|total_revenue       |total_fuel_cost   |total_distance_miles|profit       |profit_per_mile|origin_city|origin_state|destination_city|destination_state|typical_distance_miles|
+--------+-----------+--------------------+------------------+--------------------+-------------+---------------+-----------+------------+----------------+-----------------+----------------------+
|RTE00006|1460       |5053135.729999995   |1639811.7400000002|2432234             |3413323.99   |1.4            |Chicago    |IL          |Charlotte       |NC               |675                   |
|RTE00014|1487       |9073774.530000022   |1654412.9300000016|4352436             |7419361.6    |1.7            |Phoenix    |AZ          |Kansas City     |MO               |1204                  |
|RTE00035|1441 

In [0]:
display(
    gold_route_profitability
    .orderBy(F.desc("profit"))
    .select(
        "route_id",
        "total_loads",
        "total_revenue",
        "total_fuel_cost",
        "profit",
        "profit_per_mile"
    )
)

route_id,total_loads,total_revenue,total_fuel_cost,profit,profit_per_mile
RTE00016,1465,2.3884705340000078E7,1608382.3999999994,2.227632294E7,2.28
RTE00044,1410,2.3729510630000014E7,1553387.8999999973,2.217612273E7,2.44
RTE00029,1467,2.3804106610000018E7,1707728.5900000024,2.209637802E7,2.23
RTE00012,1507,2.3418565260000013E7,1669546.5599999968,2.17490187E7,2.45
RTE00046,1534,2.2776810209999975E7,1721908.0499999993,2.105490216E7,2.45
RTE00048,1525,2.2591100289999966E7,1665771.0800000003,2.092532921E7,2.43
RTE00042,1447,2.1779858830000002E7,1615945.9599999997,2.016391287E7,2.14
RTE00030,1445,1.8677476150000006E7,1644240.2000000002,1.703323595E7,2.2
RTE00021,1473,1.8313020130000044E7,1643261.260000005,1.666975887E7,2.16
RTE00003,1496,1.8156403030000012E7,1668535.8900000004,1.648786714E7,2.25


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
print("MOST PROFITABLE ROUTES")
gold_route_profitability \
    .orderBy(F.desc("profit")) \
    .select(
        "route_id",
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state",
        "profit",
        "profit_per_mile"
    ) \
    .show(5, truncate=False)

print("LEAST PROFITABLE ROUTES")
gold_route_profitability \
    .orderBy(F.asc("profit")) \
    .select(
        "route_id",
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state",
        "profit",
        "profit_per_mile"
    ) \
    .show(5, truncate=False)

MOST PROFITABLE ROUTES
+--------+------------+------------+----------------+-----------------+-------------+---------------+
|route_id|origin_city |origin_state|destination_city|destination_state|profit       |profit_per_mile|
+--------+------------+------------+----------------+-----------------+-------------+---------------+
|RTE00016|Philadelphia|PA          |Seattle         |WA               |2.227632294E7|2.28           |
|RTE00044|Charlotte   |NC          |Portland        |OR               |2.217612273E7|2.44           |
|RTE00029|Seattle     |WA          |Charlotte       |NC               |2.209637802E7|2.23           |
|RTE00012|Phoenix     |AZ          |Philadelphia    |PA               |2.17490187E7 |2.45           |
|RTE00046|Columbus    |OH          |Los Angeles     |CA               |2.105490216E7|2.45           |
+--------+------------+------------+----------------+-----------------+-------------+---------------+
only showing top 5 rows
LEAST PROFITABLE ROUTES
+--------+-